# ManGOE tutorial: Sonifying the Approach of the Gannon Storm using data from the ACE satellite

In this example, we will use the ManGOE framework to make asonification of the solar wind. It really is comparable to how wind down here works - it has a speed, it creates pressure, it varies a lot... *but it's also much faster than wind on Earth is, and it's also radioactive*.

The data we use comes from **Advanced Composition Explorer (ACE)** spacecraft sits between us and the Sun (at something called a Lagrange point, where the pull towards the Earth is the same as towards the Sun, so it gets stuck)... about 1.5 million km upstream of Earth in the solar wind. This far away, ACE acts as an early warning system — it measures the solar wind plasma and interplanetary magnetic field (IMF) *before* they hit Earth's magnetosphere. This gives cleverer people than I at the MET office time_hr to provide warnings to the UK services in case some spicy happens.

And those times do indeed happen - we'll be looking at data from **9–13 May 2024**, which captured the **Gannon superstorm** you likely saw in 2024 — the most intense geomagnetic storm since 1989. Multiple coronal mass ejections (CMEs) hit Earth and lead to a huge auroral extent, even as far south as the Caribbean.

---


## Setup and Imports

First, let's import everything we need. This should be familiar from previous sessions.

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt

#Point to relevant subroutines and data folders
import sys
sys.path.insert(0, '../subroutines/')
sys.path.insert(0, '../Data/')

# Import the ManGOE object sonification routine
from ManGOE import ManGOE_Object, ManGOE_Event

## Eyes on the Storm - Visualising the Data

Before we do anything audible, it's a good idea to see what exactly it is we're sonifying. To do this, we're going to load in the data, and then we plot these across the dats before, during and after the storm!

Our data today consists of the following tidbits:

- `time_hr` (hrs): the time that has elapsed since midnight on March 14th
- `Bz` (nanotesla): The vertical (up-down) component of the interplanetary magnetic field (IMF). Large negative values for this magnetic field data is indicative of geomagnetic storms taking place.
- `Bmag` (nanoteslas): The total strength of interplanetary magnetic field measured by the spacecraft, typically denoted as B or |B|.
- `V_SW` (km/s): The average speed of the particles streaming from the Sun to the Earth, known better as the **solar wind**.
- `N_p` (cm^-3): the number of proton, on average, per cubic centimetre seen by the spacecraft.

We load this data into this interactive notebook and visualise these datasets

In [ ]:
# Time to load some data!
time_hr, Bz, Bmag, V_SW, N_p = np.loadtxt('../Data/gannon_storm_ace_data.csv', delimiter=',', skiprows=1, unpack=True)

#Visualise it by creating a 4-by-1 set of axes
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

# Panel 1: IMF Bz
ax = axes[0]
ax.plot(time_hr / 24, Bz, 'b-', linewidth=0.5)
ax.axhline(0, color='grey', linestyle='--', linewidth=0.5)
ax.set_ylabel('IMF Bz [nT]')
ax.set_title('ACE Solar Wind Data — Gannon Storm, 9–13 May 2024')

# Panel 2: Total |B|
ax = axes[1]
ax.plot(time_hr/ 24, Bmag, 'k-', linewidth=0.5)
ax.set_ylabel('|B| [nT]')

# Panel 3: Solar wind speed
ax = axes[2]
ax.plot(time_hr/ 24, V_SW, 'r-', linewidth=0.5)
ax.set_ylabel('V_sw [km/s]')

# Panel 4: Proton density
ax = axes[3]
ax.plot(time_hr/ 24, N_p, 'g-', linewidth=0.5)
ax.set_ylabel('Np [cm⁻³]')
ax.set_xlabel('Days from 9 May 2024')
ax.set_xlim(0,5)

plt.tight_layout()
plt.show()

### Something to Think About
*Take a gander at the plots. Can you spot the ejection arrivals? What happens to the solar wind speed, density, and Bz when a CME hits? How obvious is this arrival at ACE - is it subtle, or is it wildly apparent?*

*Notice how Bz swings strongly negative — this is one of the more reliable measures of when the Earht's magnetosphere is going to have a wild ride. Negative Bz ultimately opens up the magnetosphere to the solar wind's energy and lets it all pour in... like some kind of microscopic flood!*

---

## Making our Sonifications and Providing Information to ManGOE

Now we see it, it's time to have a play around and see whether we can make a sound experience that conveys something similar!

To be really, REALLY clear - *there is no right or wrong answer here*. We're not looking for *the perfect way* and then saying all other ways suck. That's not how actual research goes! What we do is we have a play around, see what works and see what doesn't - this allows us to build up **intuition**.

And it's okay to make bad ones! It tells us what doesn't work, and we just don't do something like that in the future - and only the three of us will know about it (and Dan will have forgotten in a few days because he is OLD)

Below is a **control panel** where you decide how each solar wind parameter maps to a sound property. We'll start with the **Object Sonifications**, which makes continuous sounds.

**Available sound properties:**
- `'volume'` — higher values → louder
- `'cutoff'` — controls a low-pass filter (higher values → brighter/sharper sound)
- `'pitch_shift'` — shifts the pitch up or down by half an octave (i.e. a row of notes on a piano)

**Available solar wind parameters:**
- `Bz` — IMF Bz component (strongly negative = storm time, buddy!)
- `Bmag` — total magnetic field strength |B|
- `V_SW` — solar wind speed (in kilometers per second, which is blimmin' fast)
- `N_p` — proton number density - they are like

We use these properties and datasets to provide the sonfication platform STRAUSS in the form of a **dictionary**, by providing it with pairs as follows:

`mappings = {'sound property 1':data_1, 'sound property 2':data_2, ...}`

note that the sound property appears in quotation marks (for example `'volume'`) but the data does not (for example, Bz).

Edit the cell below to create your own mapping! Set any property to `None` (case sensitive, most things in code are!) if you don't want to use it.

In [ ]:
# Set up you mappings here

mappings = {
    'volume':Bmag,           # What drives the VOLUME?
    'cutoff':None,                    # What drives the FILTER CUTOFF?
    'pitch_shift': 1-np.sign(Bz)               # What drives the PITCH SHIFT?
}

# Set your notes, duration and sound system here

notes = [["A3"]]            # Base note(s) — try [["C3","E3","G3"]] for a chord!
length = 30                              # Duration of sonification in seconds
system = 'mono'                          # 'mono' or 'stereo'

# Define any presets to determine sound character

preset = 'windy'    # Try: 'default', 'windy', 'pitch_mapper', 'breathy'


# The system will summarise what's going into ManGOE here!
print("Sonification data set!")

for property, data in mappings.items():
    status = '✅' if data is not None else 'None'
    print(f"  {property:.<20s} {status}")

## Building the Sonification

The cell below takes your mappings, note and prescribed length and generates the sonification for you! It will then display the waveform for the sound and an audio player of your sound

In [ ]:
soni = ManGOE_Object(notes,length,mappings)

print("Sonification rendered! Let's see what we got!")

# Play it!
dobj = soni.notebook_display()

## Saving Your Sonification

Once you've found a mapping you're happy with, you can save the audio file:

In [ ]:
# Save the sonification as a WAV file
output_filename = 'gannon_storm_sonification.wav'
soni.save(output_filename)

## ManGOE_Event example - sounds at given times

The other sonification approach is to have the data map to a sound at a given instance in time - **this is event sonification**. The approach to sonifying the data is almost the same

**Available sound properties:**
- `'volume'` — higher values → louder
- `'cutoff'` — controls a low-pass filter (higher values → brighter/sharper sound)
- `'pitch'` — shifts the pitch up or down (similar to pitch_shift in the object sonification case)

**Available solar wind parameters:**
- `Bz` — IMF Bz component (strongly negative = storm time, buddy!)
- `Bmag` — total magnetic field strength |B|
- `V_SW` — solar wind speed (in kilometers per second, which is blimmin' fast)
- `N_p` — proton number density - they are like


We use these properties and datasets to provide the sonfication platform STRAUSS in the form of a **dictionary**, by providing it with pairs as follows:

`mappings = {'sound property 1':data_1, 'sound property 2':data_2, ...}`

note that the sound property appears in quotation marks (for example `'volume'`) but the data does not (for example, Bz).


The key difference in event sonifications is **you have to supply a list time times to the generator so it knows when to play**. This is done as the `'time'` mapping variable. In this example, the array `time` has this information.

Another curiosity of the events-type sonification is that the map limits for `'time'` have to exceed 100% for the sonification to render. I'm not sure why but there we go!

Another feature we can pass into event sonifications is a **downsampling rate**, which tells it to render one in every X notes instead of all of them - which is great if you have a lot of time points over a short duration. This is passed as an optional variable in a similar way to the way length is

In [ ]:
# Set up you mappings here

mappings = {
    'time': time,
    'volume':Bmag,           # What drives the VOLUME?
    'cutoff':None,                    # What drives the FILTER CUTOFF?
    'pitch': V_SW               # What drives the PITCH SHIFT?
}

map_lims = {
    'time':('0%','100%')
}

# Set your notes, duration and sound system here

notes = [["A3","C#4","E4","G4"]]            # Base note(s) — try [["C3","E3","G3"]] for a chord!
length = 30                              # Duration of sonification in seconds
system = 'mono'                          # 'mono' or 'stereo'
downsample = 60                          #Play only one in every X notes

# Define any presets to determine sound character

preset = 'windy'    # Try: 'default', 'windy', 'pitch_mapper', 'breathy'


# The system will summarise what's going into ManGOE here!
print("Sonification data set!")

for property, data in mappings.items():
    status = '✅' if data is not None else 'None'
    print(f"  {property:.<20s} {status}")

    soni = ManGOE_Event(notes,length,mappings,map_lims=map_lims,downsample=downsample)

# Play it!
dobj = soni.notebook_display()

Hopefully, this tutorial has taught you how one is able to use ManGOEs t make richer sonifications by building the sound using multiple data sources - each controlling a property of the sound generated. This is one step more complex than PLUM, in that we have to define the input maps, but once you get a handle on that the two sonification procedures operate much the same.